# 1. Data ingress

In [1]:
from pathlib import Path
import zipfile

cwd = Path.cwd()
if (cwd / "downloads").exists():
    DOWNLOADS = cwd / "downloads"
    EXTRACTED = cwd / "analysis" / "extracted"
else:
    DOWNLOADS = cwd.parent / "downloads"
    EXTRACTED = cwd / "extracted"

# Find all ZIP files (case-insensitive: .zip and .ZIP)
zip_files = list(DOWNLOADS.rglob("*.zip")) + list(DOWNLOADS.rglob("*.ZIP"))
zip_files = sorted(set(zip_files))

print(f"Found {len(zip_files)} ZIP files")

Found 55 ZIP files


In [2]:
EXTRACTED.mkdir(parents=True, exist_ok=True)
datafiles: list[Path] = []

for zip_path in zip_files:
    # Preserve date folder: downloads/05-08-2024/foo.zip -> extracted/05-08-2024/
    date_folder = zip_path.parent.name
    out_dir = EXTRACTED / date_folder
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
        for name in zf.namelist():
            if name.lower().endswith((".docx", ".doc")):
                datafiles.append(out_dir / name)

print(f"Extracted {len(datafiles)} data files to {EXTRACTED}")

Extracted 7527 data files to /home/sahir/Desktop/Roaddanger/data collection/analysis/extracted


In [3]:
# Preview first few
datafiles[:5]

[PosixPath('/home/sahir/Desktop/Roaddanger/data collection/analysis/extracted/01-01-2024/Jaarwisseling in Groningen kenmerkt zich door heftige gebeurtenissen. Grote ongeregeldheden blijven.DOCX'),
 PosixPath('/home/sahir/Desktop/Roaddanger/data collection/analysis/extracted/01-01-2024/Dode (39) en vier zwaargewonden bij botsing tegen boom in Tilburg _ ‘Ik hoor hier vaak remmen en kla.DOCX'),
 PosixPath('/home/sahir/Desktop/Roaddanger/data collection/analysis/extracted/01-01-2024/Dode (39) en vier zwaargewonden bij botsing tegen boom in Tilburg _ ‘Ik hoor hier vaak remmen en kla(2).DOCX'),
 PosixPath('/home/sahir/Desktop/Roaddanger/data collection/analysis/extracted/01-01-2024/Dode (39) en vier zwaargewonden bij botsing tegen boom in Tilburg _ ‘Ik hoor hier vaak remmen en kla(3).DOCX'),
 PosixPath('/home/sahir/Desktop/Roaddanger/data collection/analysis/extracted/01-01-2024/Dode (39) en vier zwaargewonden bij botsing tegen boom in Tilburg _ ‘Ik hoor hier vaak remmen en kla(4).DOCX')]

# 2. Extract content from DOCX

Based on the structure: Heading 1 = title, next Normal = outlet, next = date, "Length: X words", "Byline: X", body between "Body" and "Load-Date:".

In [6]:
import re
from docx import Document

def extract_article(path: Path) -> dict:
    doc = Document(path)
    title = outlet = date_str = length = author = ""
    body_parts: list[str] = []
    in_body = False

    for para in doc.paragraphs:
        text = para.text.strip()
        style = para.style.name if para.style else ""

        if style == "Heading 1":
            title = text
            continue
        if not text:
            continue

        if text.startswith("Length:"):
            m = re.search(r"(\d+)\s*words?", text, re.I)
            length = int(m.group(1)) if m else 0
            continue
        if text.startswith("Byline:"):
            author = text[7:].strip()
            continue
        if text == "Body":
            in_body = True
            continue
        if "Load-Date:" in text or text == "End of Document":
            in_body = False
            continue
        if in_body and text not in ("Link naar afbeelding",):
            body_parts.append(text)
            continue

        # Before body: outlet and date (first two Normal paras after title)
        skip = text.startswith("Copyright") or text.startswith("Length:") or text.startswith("Byline:") or text.startswith("Highlight:")
        if not skip and not in_body:
            if not outlet and title:
                outlet = text
            elif outlet and not date_str:
                date_str = text

    return {
        "title": title,
        "outlet": outlet,
        "date": date_str,
        "length": length,
        "author": author,
        "body": "\n\n".join(body_parts),
        "source_file": str(path),
        "source_date": path.parent.name,
    }

In [9]:
# Extract from first 20 files
articles = [extract_article(p) for p in datafiles]
print(f"Extracted {len(articles)} articles")

Extracted 7527 articles


In [10]:
# Preview first article
import json
print(json.dumps(articles[0], indent=2, ensure_ascii=False)[:1500])

{
  "title": "Jaarwisseling in Groningen kenmerkt zich door heftige gebeurtenissen. Grote ongeregeldheden blijven uit in Drenthe",
  "outlet": "Dagblad van het Noorden.nl",
  "date": "1 januari 2024 maandag 9:06 AM CET",
  "length": 431,
  "author": "",
  "body": "In Groningen kreeg de politie flink wat meldingen van brand, vuurwerkoverlast en vernielingen. Ook is de ME tweemaal ingezet, in Bedum en Hoogkerk. In Groningen zijn vier aanhoudingen verricht.Inzet van MEIn Bedum is de ME rond 02.10 uur kortstondig ingezet aan De Vlijt.\n\nHet was daar onrustig en er woedde brand bij een gebouw. De brandweer werd tijdens het blussen bekogeld met vuurwerk. Ook werd de politie bekogeld met vuurwerk. De ME-inzet vond plaats om een veilige werkplek te creëren voor de hulpdiensten. In Hoogkerk heeft de ME rond 03.30 uur kort opgetreden. nadat de politie bekogeld werd met stenen en vuurwerk.Explosie MidwoldaIn Midwolda heeft rond 01.10 uur aan de Anna Maria Horalaan een explosie plaatsgevonden met

In [12]:
import pandas as pd

df = pd.DataFrame(articles)
df

,title,outlet,date,length,author,body,source_file,source_date
0,Jaarwisseling in Groningen kenmerkt zich door ...,Dagblad van het Noorden.nl,1 januari 2024 maandag 9:06 AM CET,431,,In Groningen kreeg de politie flink wat meldin...,/home/sahir/Desktop/Roaddanger/data collection...,01-01-2024
1,Dode (39) en vier zwaargewonden bij botsing te...,AD/Algemeen Dagblad.nl,1 januari 2024 maandag 06:24 AM GMT,313,"Matthijs Keim, Marieke van Gompel",TILBURG - Een 39-jarige man uit Tilburg is ove...,/home/sahir/Desktop/Roaddanger/data collection...,01-01-2024
2,Dode (39) en vier zwaargewonden bij botsing te...,Brabants Dagblad.nl,1 januari 2024 maandag 06:24 AM GMT,313,"Matthijs Keim, Marieke van Gompel",TILBURG - Een 39-jarige man uit Tilburg is ove...,/home/sahir/Desktop/Roaddanger/data collection...,01-01-2024
3,Dode (39) en vier zwaargewonden bij botsing te...,BN De Stem.nl,1 januari 2024 maandag 06:24 AM GMT,313,"Matthijs Keim, Marieke van Gompel",TILBURG - Een 39-jarige man uit Tilburg is ove...,/home/sahir/Desktop/Roaddanger/data collection...,01-01-2024
4,Dode (39) en vier zwaargewonden bij botsing te...,Eindhovens Dagblad.nl,1 januari 2024 maandag 06:24 AM GMT,313,"Matthijs Keim, Marieke van Gompel",TILBURG - Een 39-jarige man uit Tilburg is ove...,/home/sahir/Desktop/Roaddanger/data collection...,01-01-2024
...,...,...,...,...,...,...,...,...
7522,Dubbele pech op spoor tussen Apeldoorn en Amer...,AD/Algemeen Dagblad.nl,31 juli 2024 woensdag 06:37 PM GMT,164,Thomas van der Kolk,Het treinverkeer tussen Apeldoorn en Amersfoor...,/home/sahir/Desktop/Roaddanger/data collection...,31-07-2024
7523,Dubbele pech op spoor tussen Apeldoorn en Amer...,Tubantia.nl,31 juli 2024 woensdag 06:37 PM GMT,163,Thomas van der Kolk,Het treinverkeer tussen Apeldoorn en Amersfoor...,/home/sahir/Desktop/Roaddanger/data collection...,31-07-2024
7524,Dubbele pech op spoor tussen Apeldoorn en Amer...,De Gelderlander.nl,31 juli 2024 woensdag 06:37 PM GMT,163,Thomas van der Kolk,Het treinverkeer tussen Apeldoorn en Amersfoor...,/home/sahir/Desktop/Roaddanger/data collection...,31-07-2024
7525,Doodeng! Man duwt voorbijganger metrorails op,De Telegraaf.nl,31 juli 2024 woensdag 7:31 PM GMT,48,,In het metrostation van Oxford Circus in Londe...,/home/sahir/Desktop/Roaddanger/data collection...,31-07-2024
